In [0]:
# Databricks notebook source
# TASK 4: Checkpoint Recovery Demonstration

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Configure isolated demo paths
# --------------------------------------------------

dbutils.widgets.text(
    "demo_root",
    "/Volumes/logistics/bronze/volume/task4_recovery_demo"
)

root = dbutils.widgets.get(
    "demo_root"
).rstrip("/")

input_path = root + "/input"

checkpoint_path = root + "/checkpoint"

output_path = root + "/output"

dbutils.fs.mkdirs(input_path)

# --------------------------------------------------
# 2. Create sample input if it does not exist
# --------------------------------------------------

sample_file = input_path + "/recovery_sample.csv"

existing_files = [
    file.path
    for file in dbutils.fs.ls(input_path)
]

if sample_file not in existing_files:

    dbutils.fs.put(
        sample_file,
        """shipment_id,event_type
S001,PICKED_UP
S002,DELIVERED
S003,AT_HUB
""",
        True
    )

# --------------------------------------------------
# 3. Define schema
# --------------------------------------------------

schema = StructType([
    StructField(
        "shipment_id",
        StringType(),
        False
    ),

    StructField(
        "event_type",
        StringType(),
        True
    )
])

# --------------------------------------------------
# 4. Read streaming input
# --------------------------------------------------

stream_df = (
    spark.readStream

    .schema(schema)

    .option("header", "true")

    .csv(input_path)

    .withColumn(
        "processed_at",
        F.current_timestamp()
    )
)

# --------------------------------------------------
# 5. Write stream with checkpoint
# --------------------------------------------------

query = (
    stream_df.writeStream

    .format("delta")

    .outputMode("append")

    .option(
        "checkpointLocation",
        checkpoint_path
    )

    .trigger(
        availableNow=True
    )

    .start(output_path)
)

query.awaitTermination()

# --------------------------------------------------
# 6. Validate output
# --------------------------------------------------

result = (
    spark.read
    .format("delta")
    .load(output_path)
)

display(
    result.orderBy("shipment_id")
)

total_rows = result.count()

distinct_ids = (
    result.select("shipment_id")
    .distinct()
    .count()
)

print("Total rows:", total_rows)

print("Distinct shipment IDs:", distinct_ids)

print("Checkpoint:", checkpoint_path)

assert total_rows == distinct_ids, (
    "Duplicate shipment IDs detected"
)

print("CHECKPOINT RECOVERY DEMO: PASS")

print(
    "Run this notebook again using the same "
    "checkpoint path and verify that no duplicate "
    "records are created."
)